In [2]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as patches
import ipywidgets as widgets
from IPython.display import display, clear_output


class MAPFReplayViewer:
    def __init__(self, trajectory, save_path="annotations.json"):
        self.trajectory = trajectory
        self.grid = trajectory["grid"]
        self.states = trajectory["states"]
        self.goals = trajectory.get("goals", [])
        self.scenario_id = trajectory.get("scenario_id", "unknown")
        self.save_path = Path(save_path)

        self.num_steps = len(self.states)
        self.num_rows = len(self.grid)
        self.num_cols = len(self.grid[0])

        self.current_timestep = 0
        self.annotations = []
        self._load_existing_annotations()
        self._build_widgets()

    def _load_existing_annotations(self):
        if self.save_path.exists():
            try:
                with open(self.save_path, "r") as f:
                    all_annotations = json.load(f)
                self.annotations = all_annotations.get(self.scenario_id, [])
            except Exception:
                self.annotations = []
        else:
            self.annotations = []

    def _save_annotations(self):
        all_annotations = {}
        if self.save_path.exists():
            try:
                with open(self.save_path, "r") as f:
                    all_annotations = json.load(f)
            except Exception:
                all_annotations = {}

        all_annotations[self.scenario_id] = self.annotations
        with open(self.save_path, "w") as f:
            json.dump(all_annotations, f, indent=2)

    def _build_widgets(self):
        self.slider = widgets.IntSlider(
            value=0,
            min=0,
            max=self.num_steps - 1,
            step=1,
            description="Timestep:",
            continuous_update=False,
            layout=widgets.Layout(width="500px")
        )
        self.slider.observe(self._on_slider_change, names="value")

        self.prev_button = widgets.Button(description="Prev")
        self.next_button = widgets.Button(description="Next")
        self.mark_button = widgets.Button(description="Mark Failure")
        self.delete_button = widgets.Button(description="Delete Mark")

        self.prev_button.on_click(self._on_prev_clicked)
        self.next_button.on_click(self._on_next_clicked)
        self.mark_button.on_click(self._on_mark_clicked)
        self.delete_button.on_click(self._on_delete_clicked)

        self.failure_type = widgets.Dropdown(
            options=["congestion", "deadlock", "symmetry", "inefficient_yield", "other"],
            value="other",
            description="Type:"
        )

        self.confidence = widgets.Dropdown(
            options=[1, 2, 3],
            value=2,
            description="Confidence:"
        )

        self.notes = widgets.Text(
            value="",
            placeholder="Optional note",
            description="Note:",
            layout=widgets.Layout(width="500px")
        )

        self.fig_out = widgets.Output()
        self.info_out = widgets.Output()

        self.controls = widgets.VBox([
            self.slider,
            widgets.HBox([self.prev_button, self.next_button, self.mark_button, self.delete_button]),
            self.failure_type,
            self.confidence,
            self.notes
        ])

    def _on_slider_change(self, change):
        self.current_timestep = change["new"]
        self._render()

    def _on_prev_clicked(self, _):
        if self.current_timestep > 0:
            self.slider.value = self.current_timestep - 1

    def _on_next_clicked(self, _):
        if self.current_timestep < self.num_steps - 1:
            self.slider.value = self.current_timestep + 1

    def _on_mark_clicked(self, _):
        entry = {
            "timestep": self.current_timestep,
            "failure_type": self.failure_type.value,
            "confidence": self.confidence.value,
            "note": self.notes.value
        }

        existing_idx = next(
            (i for i, a in enumerate(self.annotations) if a["timestep"] == self.current_timestep),
            None
        )

        if existing_idx is None:
            self.annotations.append(entry)
        else:
            self.annotations[existing_idx] = entry

        self.annotations = sorted(self.annotations, key=lambda x: x["timestep"])
        self._save_annotations()
        self._render()

    def _on_delete_clicked(self, _):
        self.annotations = [
            a for a in self.annotations if a["timestep"] != self.current_timestep
        ]
        self._save_annotations()
        self._render()

    def _render(self):
        with self.fig_out:
            clear_output(wait=True)

            fig, ax = plt.subplots(figsize=(6, 6))

            # grid
            for r in range(self.num_rows):
                for c in range(self.num_cols):
                    color = "black" if self.grid[r][c] == 1 else "white"
                    rect = patches.Rectangle((c, r), 1, 1, facecolor=color, edgecolor="gray")
                    ax.add_patch(rect)

            # goals
            for i, goal in enumerate(self.goals):
                gr, gc = goal
                ax.text(gc + 0.5, gr + 0.2, f"G{i}", ha="center", va="center", fontsize=8)

            # agents
            positions = self.states[self.current_timestep]
            for i, (r, c) in enumerate(positions):
                circle = patches.Circle((c + 0.5, r + 0.5), 0.3)
                ax.add_patch(circle)
                ax.text(c + 0.5, r + 0.5, str(i), ha="center", va="center")

            annotation = next(
                (a for a in self.annotations if a["timestep"] == self.current_timestep),
                None
            )

            title = f"{self.scenario_id} | t={self.current_timestep}/{self.num_steps - 1}"
            if annotation:
                title += f" | MARKED: {annotation['failure_type']} (conf={annotation['confidence']})"

            ax.set_title(title)
            ax.set_xlim(0, self.num_cols)
            ax.set_ylim(self.num_rows, 0)
            ax.set_aspect("equal")
            ax.set_xticks(range(self.num_cols + 1))
            ax.set_yticks(range(self.num_rows + 1))

            plt.show()

        with self.info_out:
            clear_output(wait=True)
            print(f"Current timestep: {self.current_timestep}")
            current = next(
                (a for a in self.annotations if a["timestep"] == self.current_timestep),
                None
            )
            if current:
                print("Current annotation:", current)
            else:
                print("No annotation at this timestep")

            print("\nAll annotations:")
            if self.annotations:
                for ann in self.annotations:
                    print(ann)
            else:
                print("None")

    def show(self):
        display(self.controls)
        display(self.fig_out)
        display(self.info_out)
        self._render()

In [ ]:
toy_trajectory = {
    "scenario_id": "toy_case_1",
    "grid": [
        [0, 0, 0, 1, 0],
        [0, 1, 0, 1, 0],
        [0, 0, 0, 0, 0],
        [0, 1, 1, 0, 0],
        [0, 0, 0, 0, 0],
    ],
    "states": [
        [(0, 0), (4, 4), (2, 0)],
        [(0, 1), (3, 4), (2, 1)],
        [(0, 2), (2, 4), (2, 2)],
        [(1, 2), (2, 3), (2, 2)],
        [(2, 2), (2, 2), (2, 3)],
    ]
}

viewer = MAPFReplayViewer(toy_trajectory, save_path="annotations.json")
viewer.show()

Output()

Output()

In [5]:
deadlock_scenario = {
    "scenario_id": "deadlock_toy",
    "grid": [
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
    ],
    "states": [
        [(0,1), (1,2), (2,1), (1,0)],  # t=0 (cycle around center)
        [(0,1), (1,2), (2,1), (1,0)],  # t=1 (no movement)
        [(0,1), (1,2), (2,1), (1,0)],  # t=2
        [(0,1), (1,2), (2,1), (1,0)],  # stuck forever
    ],
    "goals": [
        (1,2), (2,1), (1,0), (0,1)  # cyclic dependency
    ]
}

viewer = MAPFReplayViewer(deadlock_scenario)
viewer.show()

Output()

Output()

In [ ]:
congestion_scenario = {
    "scenario_id": "congestion_toy",
    "grid": [
        [0, 1, 0],
        [0, 1, 0],
        [0, 0, 0],
        [0, 1, 0],
        [0, 1, 0],
    ],
    "states": [
        [(0,0), (0,2), (4,0), (4,2)],  # t=0 (approaching)
        [(1,0), (1,2), (3,0), (3,2)],  # t=1 (entering funnel)
        [(2,0), (2,2), (2,1), (2,1)],  # t=2 (crowding center)
        [(2,0), (2,2), (2,1), (2,1)],  # t=3 (jam forming)
    ],
    "goals": [
        (4,2), (4,0), (0,2), (0,0)
    ]
}

viewer = MAPFReplayViewer(congestion_scenario)
viewer.show()

Output()

Output()

In [ ]:
convention_scenario = {
    "scenario_id": "convention_violation_toy",
    "grid": [
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
    ],
    "states": [
        [(1,0), (1,2)],  # t=0 (approach)
        [(1,1), (1,2)],  # t=1 (agent 0 moves forward)
        [(1,1), (1,1)],  # t=2 (collision point)
        [(1,0), (1,2)],  # t=3 (both back off)
        [(1,1), (1,2)],  # t=4 (repeat)
        [(1,1), (1,1)],  # t=5
    ],
    "goals": [
        (1,2), (1,0)
    ]
}

viewer = MAPFReplayViewer(convention_scenario)
viewer.show()

Output()

Output()